In [104]:
import nest_asyncio
nest_asyncio.apply()

import sys
sys.path.append("../")

from dotenv import load_dotenv

assert load_dotenv("../.env", override=True), "failed to load .env"

In [105]:
# !playwright install --with-deps chromium
!docker exec crystalvision-ollama-1 ollama pull $OLLAMA_CODE_MODEL

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling d7e4b00a7d7a... 100% ▕████████████████▏  15 GB                         
pulling 109037bec39c... 100% ▕████████████████▏  136 B                         
pulling 097a36493f71... 100% ▕████████████████▏ 8.4 KB                         
pulling 2490e7468436... 100% ▕████████████████▏   65 B                         
pulling d109ef1844fa... 100% ▕████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
success 


In [106]:
import os
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama

llm = ChatOllama(model=os.getenv("OLLAMA_CODE_MODEL"), temperature=0)
embedding = OllamaEmbeddings(model=os.getenv("OLLAMA_EMBED_MODEL"))

In [107]:
from crystalvision.lang.loaders import explain_database

df = explain_database()
df.head(3)

,code,rarity,cost,category_1,category_2,ex_burst,set,name_en,type_en,job_en,text_en,images,limit_break,icons,element_ja,power,element,numElements
0,1-001H,H,6,X,None,False,[Opus I],Auron,Forward,Guardian,"When Auron deals damage to your opponent, you may play 1 Fire Backup from your hand onto the field dull.",https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg,False,NaN,"(火,)",9000.0,{Fire},1
1,1-002R,R,5,X,None,False,[Opus I],Auron,Forward,Guardian,The Backups you control cannot be broken by your opponent's Summons or abilities.,https://fftcg.cdn.sewest.net/images/cards/full/1-002R_eg.jpg,False,NaN,"(火,)",9000.0,{Fire},1
2,1-003C,C,2,III,None,False,[Opus I],Red Mage,Backup,Standard Unit,《火》《1》《ダル》: Choose 1 Forward. It cannot block this turn.,https://fftcg.cdn.sewest.net/images/cards/full/1-003C_eg.jpg,False,multicard,"(火,)",NaN,{Fire},1


In [108]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [109]:
df["text_en"].sample(20)

2897                                                                                                                                                                                                                                                                                         《風》《１》, put Wingvern into the Break Zone: Choose 1 Forward of power 9000 or more. Break it.
2742                                                                                                                            When Luso enters the field, reveal the top 5 cards of your deck. Add 1 [[i]]Category FFTA2[[/]]card among them to your hand and return the other cards to the bottom of your deck in any order.
 《風》: Luso gains First Strike until the end of the turn.
3122                                                                                                               When Vivi enters the field, choose 1 Forward opponent controls. Deal it 2000 damage. If you control 4 or more [[i]]Category IX[[/]]

In [110]:
df[df['text_en'].str.contains('《LB》')].sample(1)[['name_en', 'code', 'cost', 'element', 'text_en']]

,name_en,code,cost,element,text_en
2114,Leo,22-123R,3,{Water},"(Cards with 《LB》 cannot be included in your main deck.) [[i]]Limit Break -- 1[[/]] When Leo enters the field, draw 1 card."


In [111]:
df[df['text_en'].str.contains('br')].sample(1)[['name_en', 'code', 'cost', 'element', 'text_en']]

,name_en,code,cost,element,text_en
1354,Delusory Knight,18-053C,1,{Earth},"Put Delusory Knight into the Break Zone: Choose 1 [[i]]Job Manikin[[/]]. It gains ""This Character cannot be broken."" until the end of the turn."


In [112]:
from langchain_qdrant import QdrantVectorStore

try:
    qdrant = QdrantVectorStore.from_existing_collection(
        embedding=embedding,
        collection_name="test_collection",
        # url="http://localhost:6333",
    )
except Exception as err:
    print(err)
    qdrant = None


qdrant

In [113]:
from crystalvision.lang.docs import DOCS

if qdrant is None:
    documents = []

    for document in DOCS:
        async for doc in document.alazy_load():
            documents.append(doc)

    documents

In [114]:
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient

if qdrant is None:
    vectorstore = Qdrant.from_documents(
        documents=documents,
        embedding=embedding,
        prefer_grpc=True,
        collection_name="test_collection",
    )
else:
    vectorstore = QdrantVectorStore(
        QdrantClient(prefer_grpc=True),
        collection_name="test_collection",
        embedding=embedding
    )

# Retrieve and generate using the relevant snippets of the blog.
retriever = vectorstore.as_retriever(
    # search_type="mmr",
    # search_kwargs={'lambda_mult': 1.0}
    # search_type="similarity_score_threshold",
    # search_kwargs={'score_threshold': 0.35}
)
retriever

VectorStoreRetriever(tags=['QdrantVectorStore', 'OllamaEmbeddings'], vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x7f60566d3ed0>, search_kwargs={})

In [115]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "rules_search",
    "Search for rules information about individual cards or the rules of FFTCG. For any questions about FFTCG rules, you must use this tool after looking up the relevant cards! You should search for the mechanic, code, and/or card name to find the rules.",
)

prefix = "You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. "
for col in df.columns:
    if (col_desc := df[col].attrs.get("description", "")):
        prefix += f"'{col}' refers to {col_desc}. "

prefix += "When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe or df. You can never load or read a csv. You must never make up information."

print(prefix)

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    include_df_in_prompt=None,
    allow_dangerous_code=True,
    prefix=prefix,
    extra_tools=[retriever_tool]
)

You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. 'code' refers to the UUID of the card. 'ex_burst' refers to if the card has an exburst/ex burst/ex-burst/ex/EX ability. 'name_en' refers to the English name of the card. 'type_en' refers to the English type of the card. 'images' refers to the card image URL. 'element_ja' refers to each element of the card in Japanese as python set. 'power' refers to the numerical value of power of the card. 'element' refers to each element of the card as python set. 'numElements' refers to the number of elements the card consists of. When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe or df. You can never load or read a csv. You must never make up information.


In [116]:
agent.invoke("How many Auron cards are there?")



> Entering new AgentExecutor chain...
Thought: I need to filter the DataFrame for rows where 'name_en' is 'Auron' and count them.
Action: python_repl_ast
Action Input: df[df['name_en'] == 'Auron'].shape[0]6I now know the final answer
Final Answer: There are 6 Auron cards.

> Finished chain.


{'input': 'How many Auron cards are there?',
 'output': 'There are 6 Auron cards.'}

In [117]:
agent.invoke("Are all Auron cards mono element?")



> Entering new AgentExecutor chain...
Thought: I need to check if all Auron cards have only one element.
Action: rules_search
Action Input: "Auron element" In reference to the card 20-002H - Auron: - The original source of the damage inflicted on Auron through the action ability's swapping effect is not changed. For example, if swapping damage inflicted on you by [3-147L] Zodiark Keeper of Precepts, 3-147L Zodiark Keeper of Precepts will inflict 8000 damage on Auron.

- If Auron is not present on the Field at the point when damage is resolved, no swapping will occur and you will take damage.
Serials / Codes referenced are 20-002H, 3-147L.

In reference to the card 5-148H - Kam'lanaut: Kam'lanaut's Element switching ability overwrites his current Element. (He ceases to have that Element.) You can select the same Element that he already has again.
Serials / Codes referenced are 5-148H.

In reference to the card 11-072R - Tyro: Tyro can create CP of any Element, but this does not mean t

{'input': 'Are all Auron cards mono element?',
 'output': 'Not all Auron cards are mono-element. Some Auron cards have only Fire as their element, while at least one has Water as its element.'}

In [118]:
agent.invoke("Tell me about an existing random multi element card.")



> Entering new AgentExecutor chain...
Thought: I need to find a random multi-element card in the DataFrame.
Action: python_repl_ast
Action Input: ```python
df[df['numElements'] > 1].sample(1)
```         code rarity  cost category_1 category_2  ex_burst  \
1428  18-127C      C     2         XI       None     False   

                        set    name_en  type_en  job_en  \
1428  [Resurgence of Power]  Lilisette  Forward  Dancer   

                                                                                                                                                                                                                                   text_en  \
1428  When Lilisette enters the field, choose 1 Forward of cost 2 or less opponent controls. Put it into the Break Zone.    [[s]]Sensual Dance[[/]] 《S》《ダル》: All the Forwards opponent controls lose 4000 power until the end of the turn.   

                                                             images  \
1428  https

{'input': 'Tell me about an existing random multi element card.',
 'output': 'LILISETTE [18-127C] is a Forward Dancer with 5000 power and the elements Lightning and Water. Her abilities are "When Lilisette enters the field, choose 1 Forward of cost 2 or less opponent controls. Put it into the Break Zone." and "[[s]]Sensual Dance[[/]] 《S》《ダル》: All the Forwards opponent controls lose 4000 power until the end of the turn.".'}

In [119]:
agent.invoke("Tell me about a card that consists of 3 or more elements.")



> Entering new AgentExecutor chain...
Thought: I need to find a card with 3 or more elements. I can do this by filtering the DataFrame df.
Action: python_repl_ast
Action Input: ```python
df[df['numElements'] >= 3].head()
```         code rarity  cost category_1 category_2  ex_burst                set  \
1569  19-128L      L     4          I       None     False  [From Nightmares]   

               name_en  type_en            job_en  \
1569  Warrior of Light  Forward  Warrior of Light   

                                                                                                                                                                                                                                                                text_en  \
1569  You can only pay with CP produced by Backups to cast Warrior of Light.    Haste First Strike Brave    When Warrior of Light enters the field due to your cast, activate all the Backups you control. Draw 1 card.    《ダル》: Choose 1 For

{'input': 'Tell me about a card that consists of 3 or more elements.',
 'output': 'WARRIOR OF LIGHT [19-128L] is a Forward with 4 elements: Water, Fire, Earth, and Wind.'}

In [120]:
agent.invoke("I want an Ice element card that has the highest power.")



> Entering new AgentExecutor chain...
Thought: I need to filter the DataFrame for cards with 'Ice' in their elements and then find the one with the highest power.
Action: python_repl_ast
Action Input: ```python
df[df['element'].apply(lambda x: 'Ice' in x)].sort_values('power', ascending=False).head(1)
```         code rarity  cost            category_1 category_2  ex_burst  \
2278  24-033L      L     7  MOBIUS &middot; XIII       XIII     False   

                   set     name_en  type_en job_en  \
2278  [Hidden Legends]  Bhunivelze  Forward    God   

                                                                                                                                                                                                                                                                                                                                                                                                        text_en  \
2278  When Bhunivelze enters the 

{'input': 'I want an Ice element card that has the highest power.',
 'output': 'BHUNIVELZE [24-033L] is an Ice element card with the highest power of 10000.'}

In [121]:
agent.invoke("Tell me about 1-001H.")



> Entering new AgentExecutor chain...
Thought: I need to find information about card 1-001H in the DataFrame df.
Action: python_repl_ast
Action Input: df[df['code'] == '1-001H']     code rarity  cost category_1 category_2  ex_burst       set name_en  \
0  1-001H      H     6          X       None     False  [Opus I]   Auron   

   type_en    job_en  \
0  Forward  Guardian   

                                                                                                    text_en  \
0  When Auron deals damage to your opponent, you may play 1 Fire Backup from your hand onto the field dull.   

                                                         images  limit_break  \
0  https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg        False   

  icons element_ja   power element  numElements  
0   NaN       (火,)  9000.0  {Fire}            1  I now know the final answer
Final Answer: Auron [1-001H] is a Forward unit with 9000 power and the job Guardian. It costs 6 CP to play. 

{'input': 'Tell me about 1-001H.',
 'output': 'Auron [1-001H] is a Forward unit with 9000 power and the job Guardian. It costs 6 CP to play. When Auron deals damage to your opponent, you may play 1 Fire Backup from your hand onto the field dull.  Auron has the element Fire.'}

In [122]:
agent.invoke("What are the codes for cards named 'Auron'?")



> Entering new AgentExecutor chain...
Thought: I need to find all cards with the name 'Auron' and return their codes.
Action: python_repl_ast
Action Input: df[df['name_en'] == 'Auron']['code'].tolist()['1-001H', '1-002R', '16-136S', '20-002H', '22-001R', '7-001H']I now know the final answer
Final Answer: The codes for cards named 'Auron' are ['1-001H', '1-002R', '16-136S', '20-002H', '22-001R', '7-001H'].

> Finished chain.


{'input': "What are the codes for cards named 'Auron'?",
 'output': "The codes for cards named 'Auron' are ['1-001H', '1-002R', '16-136S', '20-002H', '22-001R', '7-001H']."}

In [123]:
agent.invoke("What does each 'Auron' do?")



> Entering new AgentExecutor chain...
Thought: I need to find all cards named 'Auron'. Then, I need to look up their effects in the rules.
Action: python_repl_ast
Action Input: df[df['name_en'].str.contains('Auron')]         code rarity  cost category_1 category_2  ex_burst  \
0      1-001H      H     6          X       None     False   
1      1-002R      R     5          X       None     False   
1167  16-136S      S     2          X       None     False   
1729  20-002H      H     5          X       None     False   
1992  22-001R      R     2          X       None     False   
2969   7-001H      H     4          X       None     False   

                        set name_en  type_en    job_en  \
0                  [Opus I]   Auron  Forward  Guardian   
1                  [Opus I]   Auron  Forward  Guardian   
1167  [Emissaries of Light]   Auron  Forward  Guardian   
1729       [Dawn of Heroes]   Auron  Forward  Guardian   
1992          [Hidden Hope]   Auron   Backup  Guardian   

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `This document appears to contain information about Final Fantasy trading cards, likely from a game like Final Fantasy Trading Card Game (FFTCG). Here's a breakdown of what we see:

* **Card Descriptions:**  You have descriptions for several cards, including Balthier (Sky Pirate), Bomb, and Firion. These descriptions detail their abilities and how they function in the game.
* **Rules Clarifications:** There are also sections that clarify specific rules interactions:
    * **Firion's Double Attack:** Explains that even if Firion gains the ability to attack twice, he can only do so a maximum of two times per turn.
    * **Typhon's Transformation:**  States that if Typhon is transformed into a Forward but loses that status while attacking, the attack is canceled.
* **Deck Pl

In [124]:
agent.invoke("I need a chart showing the distribution of elements")



> Entering new AgentExecutor chain...
Thought: I need to count the occurrences of each element across all cards.
Action: python_repl_ast
Action Input: 
```python
from collections import Counter

element_counts = Counter()
for index, row in df.iterrows():
  element_counts.update(row['element'])

print(element_counts)
```Counter({'Lightning': 592, 'Fire': 580, 'Earth': 573, 'Ice': 571, 'Wind': 566, 'Water': 557, 'Dark': 62, 'Light': 58})
I now know the final answer
Final Answer: 
```
Element    Count
Lightning   592
Fire        580
Earth       573
Ice         571
Wind        566
Water       557
Dark         62
Light        58
```

> Finished chain.


{'input': 'I need a chart showing the distribution of elements',
 'output': '```\nElement    Count\nLightning   592\nFire        580\nEarth       573\nIce         571\nWind        566\nWater       557\nDark         62\nLight        58\n```'}

In [125]:
output = agent.invoke("I want the lowest power forward that also has an ex burst.")["output"]
print(output)
agent.invoke(output + " What is its EX ability (ex abilities can be parsed from text_en)?")



> Entering new AgentExecutor chain...
Thought: I need to filter the DataFrame for forwards with an ex burst and then find the one with the lowest power.
Action: python_repl_ast
Action Input: ```python
df[df['type_en'] == 'Forward'][df['ex_burst'] == True]['power'].min()
```2000.0

<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


I now know the final answer
Final Answer: The lowest power forward with an ex burst has a power of 2000.

> Finished chain.
The lowest power forward with an ex burst has a power of 2000.


> Entering new AgentExecutor chain...
Thought: I need to find all forwards with an ex burst and then find the one with the lowest power.
Action: python_repl_ast
Action Input: df[df['type_en'] == 'Forward'][df['ex_burst'] == True].sort_values('power').head(1)        code rarity  cost              category_1 category_2  ex_burst  \
3369  9-115R      R     2  PICTLOGICA &middot; IV         IV      True   

            set name_en  type_en      job_en  \
3369  [Opus IX]   Porom  Forward  White Mage   

                                                                                                                                 text_en  \
3369  [[ex]]EX BURST[[/]] When Porom is put from the field into the Break Zone, choose 1 Summon in your Break Zone. Add it to your hand.   

                          

<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


I now know the final answer
Final Answer: When Porom is put from the field into the Break Zone, choose 1 Summon in your Break Zone. Add it to your hand. 


> Finished chain.


{'input': 'The lowest power forward with an ex burst has a power of 2000. What is its EX ability (ex abilities can be parsed from text_en)?',
 'output': 'When Porom is put from the field into the Break Zone, choose 1 Summon in your Break Zone. Add it to your hand.'}

In [126]:
output = agent.invoke("Pick a card at random whose type is backup but also has an ex burst.")["output"]
print(output)
agent.invoke(output + " What is its EX ability (ex abilities can be parsed from text_en)?")



> Entering new AgentExecutor chain...
Thought: I need to find a Backup card with an EX Burst ability. I can use pandas to filter the DataFrame for this.
Action: python_repl_ast
Action Input: ```python
df[(df['type_en'] == 'Backup') & (df['ex_burst'] == True)].sample(n=1)
```        code rarity  cost category_1 category_2  ex_burst       set name_en  \
236  10-021C      C     3       WOFF       None      True  [Opus X]  Rorrik   

    type_en  job_en  \
236  Backup  Father   

                                                                                                                                                       text_en  \
236  [[ex]]EX BURST[[/]] When Rorrik enters the field, choose 1 [[i]]Card Name Lann[[/]] or [[i]]Card Name Reynn[[/]] in your Break Zone. Add it to your hand.   

                                                            images  \
236  https://fftcg.cdn.sewest.net/images/cards/full/10-021C_eg.jpg   

     limit_break     icons element_ja  power elemen

{'input': 'Rorrik [10-021C] is a Backup card with an EX Burst. What is its EX ability (ex abilities can be parsed from text_en)?',
 'output': 'When Rorrik enters the field, choose 1 Card Name Lann or Card Name Reynn in your Break Zone. Add it to your hand.'}

In [127]:
agent.invoke("Tell me about paying crystal(s) and the Ice element card named Lady Lilith.")



> Entering new AgentExecutor chain...
Thought: I need to find information about Crystal ([C]) payment and the card Lady Lilith.
Action: rules_search
Action Input: "Crystal payment"In reference to the card 16-013H - Sol: Crystal is paid at the point of resolution. After it is paid, the auto-ability “Play 1 Forward of cost 4 or less from your hand onto the field” is placed onto the stack.
Serials / Codes referenced are 16-013H.

In reference to the card 15-084L - Robel-Akbel: Crystal is paid at the point of resolution. After it has been paid, the auto-ability “Choose 1 Forward or Monster. Break it.” goes onto the stack.
Serials / Codes referenced are 15-084L.

In reference to the card [24-047R] Sophia (SOPFFO): {crystalicon} is paid at the point when effects are resolved. The Auto Ability to select one option from the list below is then added to the Stack after the payment is made.
Serials / Codes referenced are 24-047R.

In reference to the card [23-038H] Lady Lilith: The Crystal is p

{'input': 'Tell me about paying crystal(s) and the Ice element card named Lady Lilith.',
 'output': 'Crystal ([C]) is paid at the point when effects are resolved for Lady Lilith [23-038H]. After paying Crystal, her auto ability to select one Forward and gain control of it is added to the stack.'}

In [128]:
agent.invoke("Tell me about the card with code/serial 19-138S.")



> Entering new AgentExecutor chain...
Thought: I need to find the card with the given code in the DataFrame.
Action: python_repl_ast
Action Input: df[df['code'] == '19-138S']         code rarity  cost category_1 category_2  ex_burst                set  \
1579  19-138S      S     2       XIII       None     False  [From Nightmares]   

        name_en  type_en         job_en  \
1579  Lightning  Forward  Ravager/L'Cie   

                                                                                                                                                                                                                                                                                                                                                                            text_en  \
1579  Haste    When 1 or more [[i]]Category XIII[[/]] Forwards you control attack, all the Forwards you control gain +1000 power until the end of the turn. If 2 or more [[i]]Category XIII [[/]]Forwar

{'input': 'Tell me about the card with code/serial 19-138S.',
 'output': 'Lightning [19-138S] is a Forward card with 7000 power and the element Light. It has the ability "Haste When 1 or more Category XIII Forwards you control attack, all the Forwards you control gain +1000 power until the end of the turn. If 2 or more Category XIII Forwards were attacking this turn, also draw 1 card. If 3 or more Category XIII Forwards were attacking this turn, Lightning also deals your opponent 1 point of damage."'}

In [129]:
agent.invoke("While controlling 19-138S Lightning, what happens if I party attack with exactly 2 Category XIII forwards?")



> Entering new AgentExecutor chain...
Thought: I need to understand the rules regarding Party Attacks and Category XIII Forwards.
Action: rules_search
Action Input: "Party Attack" "Category XIII"
In reference to the card 15-132S - Jessie: The auto-ability which is activated by the party attack will be activated if two or more [Job (Avalanche)] Forwards are included in it, even if Forwards which have other Jobs are also part of it. For example, the ability can be activated by a party attack of three cards, with two [Job (Avalanche)] cards and one [Job (Merc)] card.

Jessie does not need to be included in the party attack needed to activate the auto-ability.

For German: [1-204S], [4-082C] Jesse and [15-132S] Jessie are the same character, though their names have changed.
Serials / Codes referenced are 15-132S.

In reference to the card 15-131S - Wedge: The auto-ability which is activated by the party attack will be activated if two or more [Job (Avalanche)] Forwards are included in it

{'input': 'While controlling 19-138S Lightning, what happens if I party attack with exactly 2 Category XIII forwards?',
 'output': 'Yes, while controlling 19-138S Lightning, party attacking with exactly two Category XIII forwards would likely trigger a Party Attack ability that requires two or more Forwards of the same Job.'}

In [130]:
agent.invoke("While controlling 19-138S Lightning, what happens if I party attack with exactly 2 Category XII forwards?")



> Entering new AgentExecutor chain...
Thought: I need to understand how Party Attacks work and if there are any specific rules regarding Category XII forwards.
Action: rules_search
Action Input: "Party Attack FFTCG"10 
 
10.1.1.2.  The turn player then gains priority. Each player may use a Summon, an action 
ability or a special ability.  
 
10.1.2.  Attack Declaration Step  
 
10.1.2.1.  The turn player declares one of their Forwards to make an attack, or that they 
will choose to form a party. To form a party, they must choose multiple Forwards 
that share the same Element. The player can choose Forwards that fulfil the 
following criteria:  
 
10.1.2.1.1.  The Forwards chosen  have to be active and they need to have the Haste 
attribute or have been continuously under the control of the turn player 
from the beginning of the turn.  
 
10.1.2.1.2.  They are legally able to attack (e.g. they have not already attacked this 
turn, and no effects prevent them from attacking).  
 
10.1.

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `You've got a good grasp of how parties work in FFXIV TCG! Your explanation of Zeid's effect and the First Strike interaction with a party are both accurate.  

Here are some additional thoughts to consider:

* **Party Flexibility:** Parties offer a lot of tactical flexibility. You can combine Forwards with different abilities to create powerful synergies. For example, you could pair a Forward with First Strike with another that has a strong on-hit effect.
* **Blocking Considerations:** Remember that a single blocker can block an entire party if it meets the blocking conditions for *at least one* member of the party. This means you need to carefully consider your opponent's blockers when deciding which Forwards to include in a party.
* **Counterplay:**  Opponents have way

In [131]:
agent.invoke("Tell me about paying crystal(s) for the ability on the card 23-038H. What is the rule?")



> Entering new AgentExecutor chain...
Thought: I need to find the card and then look up the rules for paying crystals.
Action: rules_search
Action Input: 'crystal cost'In reference to the card 16-013H - Sol: Crystal is paid at the point of resolution. After it is paid, the auto-ability “Play 1 Forward of cost 4 or less from your hand onto the field” is placed onto the stack.
Serials / Codes referenced are 16-013H.

In reference to the card 21-128S - Shantotto: - Crystals are paid at the point when effects are resolved. The Auto Ability “Select one “category DFF” Forward with a cost of five or lower that is present in your Break Zone and deploy it to the Field” is then added to the stack after payment is made.
Serials / Codes referenced are 21-128S.

In reference to the card 15-122L - Mog (VI): Crystal is paid at the point of resolution. After it has been paid, the auto-ability “Draw 2 cards, then discard 1 card from your hand” goes onto the stack.
Serials / Codes referenced are 15-12

{'input': 'Tell me about paying crystal(s) for the ability on the card 23-038H. What is the rule?',
 'output': 'Crystals are paid at the point of resolution.'}

In [ ]:
agent.invoke("Tell me the rules about paying crystal(s) and the card 23-038H.")



> Entering new AgentExecutor chain...
Thought: I need to find the rules about paying crystals in FFTCG and then look up the specific card 23-038H to see if it has any special rules regarding crystal payment.
Action: rules_search
Action Input: "paying crystals"In reference to the card 20-017R - Palom: - Crystals are paid at the point effects are resolved. The Auto-ability “Select one Forward and inflict 8000 damage on it” is added to the stack after paying,
Serials / Codes referenced are 20-017R.

In reference to the card 20-017R - Palom: - Crystals are paid at the point effects are resolved. The Auto-ability “Select one Forward and inflict 8000 damage on it” is added to the stack after paying,
Serials / Codes referenced are 20-017R.

In reference to the card 16-013H - Sol: Crystal is paid at the point of resolution. After it is paid, the auto-ability “Play 1 Forward of cost 4 or less from your hand onto the field” is placed onto the stack.
Serials / Codes referenced are 16-013H.

In 

{'input': 'Tell me the rules about paying crystal(s) and the card 23-038H.',
 'output': 'Crystals for card 23-038H are likely paid at the point its effect resolves, following the general FFTCG rules.'}

In [ ]:
agent.invoke("If I play Lasswell 16-042R on a field with no other characters present, does it have to dull and freeze itself?")



> Entering new AgentExecutor chain...
Thought: I need to look up the rules for Lasswell 16-042R.
Action: rules_search
Action Input: "Lasswell 16-042R"In reference to the card 16-042R - Lasswell: The auto-ability which occurs at the end of the turn will activate even if Lasswell is placed onto the field after Job Knight attacks.
Serials / Codes referenced are 16-042R.

In reference to the card 16-042R - Lasswell: The auto-ability which occurs at the end of the turn will activate even if Lasswell is placed onto the field after Job Knight attacks.
Serials / Codes referenced are 16-042R.

In reference to the card 22-113L - Mont Leonis: - If neither a Fire Forward of cost 3 or less or a Fire Forward of cost 5 or less cannot be chosen, the auto-ability is not placed on the stack.

- If you control 4 or less Backups, you also do not put a Backup into the Break Zone.
Serials / Codes referenced are 22-113L.

In reference to the card 22-113L - Mont Leonis: - If neither a Fire Forward of cost 3

{'input': 'If I play Lasswell 16-042R on a field with no other characters present, does it have to dull and freeze itself?',
 'output': 'No, Lasswell 16-042R does not have to dull and freeze itself if it is the only character on the field.'}

In [65]:
agent.invoke("Tell me about a random card with a '《ダル》' ability in its text_en. I want its name, code, cost, element, and exact text.")



> Entering new AgentExecutor chain...
Thought: I need to find a card with "《ダル》" in its text_en. I can use the DataFrame df for this.
Action: python_repl_ast
Action Input: ```python
df[df['text_en'].str.contains('《ダル》')].sample(1)[['name_en', 'code', 'cost', 'element', 'text_en']]
```    name_en     code  cost      element  \
447   Crowe  11-092C     3  {Lightning}   

                                                                                                                                                                                text_en  
447  When Crowe is put from the field into the Break Zone, choose 1 Forward in your Break Zone. Add it to your hand.    《ダル》, put Crowe into the Break Zone: Choose 1 Forward. Dull it.  I now know the final answer
Final Answer: The card is Crowe [11-092C]. It costs 3 CP and has the element Lightning. Its text is "When Crowe is put from the field into the Break Zone, choose 1 Forward in your Break Zone. Add it to your hand.    《ダル》, put C

{'input': "Tell me about a random card with a '《ダル》' ability in its text_en. I want its name, code, cost, element, and exact text.",
 'output': 'The card is Crowe [11-092C]. It costs 3 CP and has the element Lightning. Its text is "When Crowe is put from the field into the Break Zone, choose 1 Forward in your Break Zone. Add it to your hand.\u2029   《ダル》, put Crowe into the Break Zone: Choose 1 Forward. Dull it."'}